# Challenge 5: Designing a Dashboard-Ready Dataset

You've been given a **star schema** in the Silver layer and asked to prepare dashboard-ready datasets for the BI team. Your goal: create a **Gold view**, a **metric view**, and write **dashboard-quality queries**.

You'll practice:
- Writing JOIN queries against a star schema
- Creating a Gold view that hides JOIN complexity
- Creating a metric view with reusable business measures
- Querying metric views with `MEASURE()` syntax
- Applying dashboard query best practices

> **Instructions:** Run the setup cell below, then open the **SQL Editor** (left sidebar) and complete each task there. Replace `<FILL_IN>` with the correct SQL.

---

In [0]:
%run ./challenge_5_setup

## Task 1: Query the Star Schema with JOINs

Write a query that answers: **"Total revenue by customer segment and product category for Q3 2024 (July-September)."**

Revenue = `quantity * unit_price - discount_amt`

You'll need to JOIN `fact_orders` to `dim_customers` and `dim_products`.

---

In [0]:
%sql
-- Task 1: Total revenue by customer segment and product category, Q3 2024
-- JOIN fact_orders to dim_customers and dim_products

SELECT
  c.<FILL_IN> AS customer_segment,
  p.<FILL_IN> AS product_category,
  ROUND(SUM(f.quantity * f.unit_price - f.discount_amt), 2) AS total_revenue
FROM fact_orders f
<FILL_IN> dim_customers c ON f.customer_id = c.customer_id
<FILL_IN> dim_products p ON f.product_id = p.product_id
WHERE f.order_date >= '2024-07-01'
  AND f.order_date < '2024-10-01'
GROUP BY <FILL_IN>
ORDER BY total_revenue DESC;

## Task 2: Create a Gold View Over the Star Schema

Create a view called `v_orders_gold` that pre-joins ALL dimension tables to the fact table and includes:
- Calculated columns: `net_revenue` (quantity * unit_price - discount_amt), `sale_year`, `sale_quarter`
- Renamed columns for business readability (e.g., `customer_segment` not `segment`)

This view will serve as the single dataset for dashboard authors — no JOINs needed downstream.

---

In [0]:
%sql
-- Task 2: Create a Gold view with pre-joined dimensions and calculated fields

CREATE OR REPLACE VIEW v_orders_gold AS
SELECT
  f.order_id,
  f.order_date,
  YEAR(f.order_date) AS sale_year,
  QUARTER(f.order_date) AS sale_quarter,
  c.customer_name,
  c.<FILL_IN> AS customer_segment,
  c.<FILL_IN> AS customer_region,
  p.product_name,
  p.<FILL_IN> AS product_category,
  p.subcategory AS product_subcategory,
  s.<FILL_IN> AS store_name,
  s.<FILL_IN> AS store_channel,
  f.quantity,
  f.unit_price,
  f.discount_amt,
  ROUND(<FILL_IN>, 2) AS net_revenue
FROM fact_orders f
INNER JOIN dim_customers c ON f.customer_id = c.customer_id
INNER JOIN dim_products p ON f.product_id = p.product_id
INNER JOIN <FILL_IN> s ON f.store_id = s.store_id;

## Task 3: Query the Gold View (Dashboard-Style)

Now query your `v_orders_gold` view to answer the same business question as Task 1 — but **without any JOINs**. Notice how much simpler the query is!

---

In [0]:
%sql
-- Task 3: Same question (revenue by segment + category, Q3 2024)
-- but from the Gold view — no JOINs needed!

SELECT
  <FILL_IN>,
  <FILL_IN>,
  ROUND(SUM(<FILL_IN>), 2) AS total_revenue
FROM v_orders_gold
WHERE order_date >= '2024-07-01'
  AND order_date < '2024-10-01'
GROUP BY customer_segment, product_category
ORDER BY total_revenue DESC;

## Task 4: Create a Metric View

Create a metric view called `orders_kpis` over the `v_orders_gold` view you just created. Define:

**Dimensions:** customer_segment, product_category, store_channel, order_date

**Measures:**
- `Total Revenue` — `SUM(net_revenue)`
- `Order Count` — `COUNT(order_id)`
- `Avg Order Value` — `SUM(net_revenue) / COUNT(order_id)`
- `Total Units` — `SUM(quantity)`

---

In [0]:
%sql
-- Task 4: Create a metric view with reusable KPIs

CREATE OR REPLACE VIEW orders_kpis
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: <FILL_IN>
  dimensions:
    - name: Customer Segment
      expr: <FILL_IN>
    - name: Product Category
      expr: <FILL_IN>
    - name: Store Channel
      expr: <FILL_IN>
    - name: Order Date
      expr: <FILL_IN>
  measures:
    - name: Total Revenue
      expr: <FILL_IN>
    - name: Order Count
      expr: <FILL_IN>
    - name: Avg Order Value
      expr: <FILL_IN>
    - name: Total Units
      expr: <FILL_IN>
$$;

## Task 5: Query the Metric View — Revenue by Segment

Query your `orders_kpis` metric view to show **Total Revenue and Order Count by Customer Segment**.

Remember the rules:
- Wrap measures with `MEASURE()`
- Use backticks around names
- Always include `GROUP BY ALL`

---

In [0]:
%sql
-- Task 5: Query the metric view — revenue and order count by segment

SELECT
  <FILL_IN>,
  MEASURE(<FILL_IN>) AS `Total Revenue`,
  MEASURE(<FILL_IN>) AS `Order Count`
FROM orders_kpis
<FILL_IN>
ORDER BY `Total Revenue` DESC;

## Task 6: Query the Metric View — Monthly Trend by Channel

Query `orders_kpis` to show a **monthly revenue trend grouped by Store Channel** for 2024.

Use `date_trunc('MONTH', ...)` for the time grouping.

---

In [0]:
%sql
-- Task 6: Monthly revenue by channel (time-series from metric view)

SELECT
  date_trunc('MONTH', <FILL_IN>) AS `Month`,
  <FILL_IN>,
  MEASURE(`Total Revenue`) AS `Total Revenue`
FROM orders_kpis
WHERE YEAR(`Order Date`) = 2024
GROUP BY ALL
ORDER BY <FILL_IN> ASC;

## Task 7: Write a Dashboard-Ready Query

Write a query from `v_orders_gold` that a dashboard widget could use to show:

**"Top 5 products by total revenue for Enterprise customers in Q4 2024, via the online channel"**

Follow best practices:
- Select only the columns you need
- Filter early and specifically
- Use clear column aliases
- Include LIMIT
- No JOINs (use the Gold view)

---

In [0]:
%sql
-- Task 7: Dashboard-ready query
-- Top 5 products by revenue, Enterprise segment, Q4 2024, online channel

SELECT
  <FILL_IN> AS product_name,
  ROUND(SUM(<FILL_IN>), 2) AS total_revenue
FROM <FILL_IN>
WHERE <FILL_IN>
  AND <FILL_IN>
  AND <FILL_IN>
GROUP BY <FILL_IN>
ORDER BY total_revenue DESC
LIMIT <FILL_IN>;

---

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# STOP HERE — Solutions Below

Only scroll down after you've attempted all 7 tasks!

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

---

# Solutions

---

In [0]:
%sql
-- Solution: Task 1 - Star Schema JOIN query
SELECT
  c.segment AS customer_segment,
  p.category AS product_category,
  ROUND(SUM(f.quantity * f.unit_price - f.discount_amt), 2) AS total_revenue
FROM fact_orders f
INNER JOIN dim_customers c ON f.customer_id = c.customer_id
INNER JOIN dim_products p ON f.product_id = p.product_id
WHERE f.order_date >= '2024-07-01'
  AND f.order_date < '2024-10-01'
GROUP BY c.segment, p.category
ORDER BY total_revenue DESC;

In [0]:
%sql
-- Solution: Task 2 - Create Gold View
CREATE OR REPLACE VIEW v_orders_gold AS
SELECT
  f.order_id,
  f.order_date,
  YEAR(f.order_date) AS sale_year,
  QUARTER(f.order_date) AS sale_quarter,
  c.customer_name,
  c.segment AS customer_segment,
  c.region AS customer_region,
  p.product_name,
  p.category AS product_category,
  p.subcategory AS product_subcategory,
  s.store_name AS store_name,
  s.channel AS store_channel,
  f.quantity,
  f.unit_price,
  f.discount_amt,
  ROUND(f.quantity * f.unit_price - f.discount_amt, 2) AS net_revenue
FROM fact_orders f
INNER JOIN dim_customers c ON f.customer_id = c.customer_id
INNER JOIN dim_products p ON f.product_id = p.product_id
INNER JOIN dim_stores s ON f.store_id = s.store_id;

In [0]:
%sql
-- Solution: Task 3 - Query the Gold View (no JOINs)
SELECT
  customer_segment,
  product_category,
  ROUND(SUM(net_revenue), 2) AS total_revenue
FROM v_orders_gold
WHERE order_date >= '2024-07-01'
  AND order_date < '2024-10-01'
GROUP BY customer_segment, product_category
ORDER BY total_revenue DESC;

In [0]:
%sql
-- Solution: Task 4 - Create Metric View
CREATE OR REPLACE VIEW orders_kpis
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: v_orders_gold
  dimensions:
    - name: Customer Segment
      expr: customer_segment
    - name: Product Category
      expr: product_category
    - name: Store Channel
      expr: store_channel
    - name: Order Date
      expr: order_date
  measures:
    - name: Total Revenue
      expr: SUM(net_revenue)
    - name: Order Count
      expr: COUNT(order_id)
    - name: Avg Order Value
      expr: SUM(net_revenue) / COUNT(order_id)
    - name: Total Units
      expr: SUM(quantity)
$$;

In [0]:
%sql
-- Solution: Task 5 - Query metric view (revenue by segment)
SELECT
  `Customer Segment`,
  MEASURE(`Total Revenue`) AS `Total Revenue`,
  MEASURE(`Order Count`) AS `Order Count`
FROM orders_kpis
GROUP BY ALL
ORDER BY `Total Revenue` DESC;

In [0]:
%sql
-- Solution: Task 6 - Monthly revenue by channel
SELECT
  date_trunc('MONTH', `Order Date`) AS `Month`,
  `Store Channel`,
  MEASURE(`Total Revenue`) AS `Total Revenue`
FROM orders_kpis
WHERE YEAR(`Order Date`) = 2024
GROUP BY ALL
ORDER BY `Month` ASC;

In [0]:
%sql
-- Solution: Task 7 - Dashboard-ready query
SELECT
  product_name,
  ROUND(SUM(net_revenue), 2) AS total_revenue
FROM v_orders_gold
WHERE customer_segment = 'Enterprise'
  AND sale_quarter = 4
  AND sale_year = 2024
  AND store_channel = 'online'
GROUP BY product_name
ORDER BY total_revenue DESC
LIMIT 5;

-- Why this is dashboard-ready:
-- 1. Only 2 columns selected (product_name + revenue)
-- 2. Specific filters enable data skipping
-- 3. No JOINs (uses Gold view)
-- 4. LIMIT 5 prevents scanning full result
-- 5. Clear column alias (total_revenue)